# ShapeSense: final pipeline

Runs top to bottom. One section per stage. Set `DATA_DIR` to the folder with the raw images (`data_set/`), which is not in the repository.

Colab link: <https://colab.research.google.com/github/Silent0Wings/ShapeSense/blob/main/notebooks/shapesense_pipeline.ipynb>

## 0. Runtime check (needs a CUDA GPU)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

## 1. Get the code and install pins

In [ ]:
!git clone https://github.com/Silent0Wings/ShapeSense.git
%cd ShapeSense
!pip install -q -r requirements.txt

## 2. Raw data

Mount Drive and link the raw images. The raw data is about 35 MB and is listed in `data/README.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ShapeSense/data_set'  # TODO: point at your copy
!ln -sfn "$DATA_DIR" data_set
!ls data_set | head

## 3. Guards and contract tests

In [ ]:
!python run_pipeline.py --check

## 4. Stage 1: data freeze and splits

Builds the unified dataset, applies the frozen cleaning rule (`exclude_source_labels` in `configs/pipeline.yaml`: Quick, Draw! diamonds), removes exact duplicates and blank images, and writes the group-aware splits. The raw baseline in section 8 is rebuilt from the raw files by replaying this step.

In [ ]:
!python run_pipeline.py --stages 1

## 5. Stage 2: augmentation (transformation)

Class-safe augmentation. F1 after this step: see section 8.

In [ ]:
!python run_pipeline.py --stages 2

## 6. Stage 3: synthetic generator

In [ ]:
!python run_pipeline.py --stages 3

## 7. Stage 4 and 5: training and test evaluation

Training is per arm and per seed. Test is read once per run (see `artifacts/test_access.log`).

In [ ]:
!python run_pipeline.py --stages 4 5

## 8. Baseline and improvement after each data preparation step

Every row trains the same CNN (same settings, 49 steps x 60 epochs, seeds 0-4) and is scored by **macro F1 on the same 2,776 test images**. Only the data preparation changes. `scripts/improvement_table.py` documents each row.

1. `replay` rebuilds every image from its raw file, only decoded and resized to 64x64.
2. `train` trains the baseline (raw, uncleaned) and the cleaning-only model.
3. `fs` picks the input resolution on validation (feature selection).
4. `evaluate` reads the test split once, through Stage 5's logged reader.

In [ ]:
!python -m scripts.improvement_table replay
!python -m scripts.improvement_table train
!python -m scripts.improvement_table fs
!python -m scripts.improvement_table evaluate

import json
table = json.load(open('results/improvement_table.json'))
rows = table['rows']
def show(i):
    r = rows[i]
    print(r['step'])
    print(f"  test macro F1 {r['mean']:.2f} +/- {r['std']:.2f} (5 seeds), delta vs baseline {r['delta_vs_baseline']:+.2f}")


### 8.1 Baseline: the collected data before any preparation

In [ ]:
show(0)

### 8.2 After cleaning

In [ ]:
show(1)

### 8.3 After transformation (standardization, then augmentation + synthetic shapes)

In [ ]:
show(2); show(3)

### 8.4 After feature selection

In [ ]:
show(4)
print(json.dumps(table['feature_selection'], indent=2))

## 9. Stage 6: report

In [ ]:
!python run_pipeline.py --stages 6

## 10. Final model vs. direct LLM prediction on the same test images

200 test images (20 per class, seed 0, `results/direct_llm_prediction/sample_ids.csv`) were shown blind to Claude Opus 5.5 and Gemini 4 Neon. The trained CNN (A2_0) is scored on the same images with the same metric. The baseline model's score on these 200 images is printed too.

In [ ]:
import csv, numpy as np
from sklearn.metrics import f1_score
def macro(path):
    rows = list(csv.DictReader(open(path)))
    return 100 * f1_score([r['true_label'] for r in rows], [r['predicted_label'] for r in rows], average='macro')
for name in ['trained_cnn', 'claude_opus', 'gemini_4_neon']:
    print(f"{name:14s} macro F1 {macro(f'results/direct_llm_prediction/{name}_predictions.csv'):.2f}")
sub = json.load(open('results/improvement_table.json'))['direct_llm_subset']
print(f"baseline (raw) macro F1 {sub['baseline_0']:.2f}; final A2_0 recomputed {sub['final_A2_0']:.2f}")
